# Workflow development

This is a little playground for me to develop functions that will be used in the final workflow.

## Imports

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw, PandasTools
import pandas as pd
import numpy as np
import MDAnalysis as mda
import nglview as nv
import os
from openmm.app import PDBFile
import requests
from pdbfixer import PDBFixer
from vina import Vina
import useful_rdkit_utils as uru

## Protein preparation

**NOTE TO SELF**: Should look into replacing `os.system` with `os.subprocess` - apparently that's safer and more professional.

In [ ]:
def fix_protein(input_protein_path:str):
    """Runs PDBFixer on the input protein structure to fix any issues such as missing residues, missing atoms, and nonstandard residues.
    The fixed structure is then saved to an internally specified output path (fixed_{input_protein_path}).

    Args:
        input_protein_path (str): The path to the input protein PDB file

    Returns:
        str: The path where the fixed protein PDB file has been saved.
    """
    # Initialise PDBFixer with input protein structure
    fixer = PDBFixer(filename=input_protein_path)

    # Identify problems with the structure and fix them
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.findNonstandardResidues()
    fixer.removeHeterogens(keepWater=False) # remove anything that isn't protein, including water and ligand(s).

    # Fix identified problems
    fixer.addMissingAtoms()
    fixer.replaceNonstandardResidues()

    # Save the fixed structure to the following output path
    fixed_protein_output_path = "fixed_"+input_protein_path
    with open(fixed_protein_output_path, 'w') as f:
        # Toplology, Positions, file stream, and keep chain ID's
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    
    return fixed_protein_output_path

def protonate_protein(input_protein_path:str, pH=7.4):
    """Runs pdb2pqr on the input protein structure to add hydrogens and assign protonated states based on the specified pH.
    The protonated structure is then saved to an internally specified output path.

    Args:
        input_protein_path (str): The path to the input protein PDB file
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).

    Returns:
        str: The path where the protonated protein .pqr file has been saved.
    """

    # Defining output path and creating protonated files. A protonated .pdb file is created primarily for future visualisation.
    protonated_protein_output_path = input_protein_path.split(".")[0]+"protonated"
    os.system(f"pdb2pqr --pdb-output={protonated_protein_output_path}.pdb --pH={pH} {input_protein_path} {protonated_protein_output_path}.pqr --whitespace")

    return f"{protonated_protein_output_path}.pqr"

def pdb_to_pdbqt(input_protein_pqr_path:str, protein_pdbqt_path:str):
    """Writes a .pdbqt file ready for Autodock Vina given a .pqr generated by pdb2pqr.

    Args:
        input_protein_pqr_path (str): The path to the input protein .pqr file (created by pdb2pqr).
        protein_pdbqt_path (str): The path where the .pdbqt file will be saved.
    """

    u = mda.Universe(f"{input_protein_pqr_path}.pqr")
    u.atoms.write(protein_pdbqt_path)

    # Read in the just-written PDBQT file, replace text, and write back
    with open(protein_pdbqt_path, 'r') as file:
        file_content = file.read()

    # Replace 'TITLE' and 'CRYST1' with 'REMARK'
    file_content = file_content.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

    # Write the modified content back to the file
    with open(protein_pdbqt_path, 'w') as file:
        file.write(file_content)

# Final function which will prepare the protein structure files with one function call.
def prepare_protein(input_protein_path:str, output_protein_pdbqt_path:str, pH=7.4): # THIS FUNCTION AND FUNCTIONS WITHIN THEM REQUIRE INPUT
    """Convenience function which fixes a protein structure with PDBFixer if any issues are present, then protonates the protein with
    pdb2pqr (propka under the hood) at a given pH, and converts the protonated structure file (.pqr used in this case) to a .pdbqt file
    ready for docking with Autodock Vina.

    Args:
        input_protein_path (str): Input protein PDB filepath.
        output_protein_pdbqt_path (str): Output protein .pdbqt filepath for Autodock Vina.
        pH (float, optional): The pH at which to assign protonated states. Defaults to 7.4 (physiological pH).
    """

    fixed_protein_path = fix_protein(input_protein_path)
    protonated_protein_pqr_path = protonate_protein(fixed_protein_path, pH=pH)
    pdb_to_pdbqt(protonated_protein_pqr_path, output_protein_pdbqt_path)


## Ligand preparation

## Docking setup

## ANNalog generation setup

## Defining directories

In [ ]:
workflow_dir = "workflow_output"
round_no = 0 # this is a counter that will be added to as generation and docking rounds are completed

generation_dir = os.path.join(workflow_dir, "generation", f"round_{round_no}")


os.makedirs(workflow_dir, exist_ok=True)